# S3 — Pandas 轉換：groupby / merge / pivot

> **時間**：2 小時  
> **資料**：`orders_clean.csv`（S2 產出）+ `customers.csv` + `products.csv`  
> **學完能幹嘛**：用 SQL 的思維在 Pandas 中做三表 join、分組統計、樞紐分析

---

## 上節回顧 + 本節為什麼重要

S2 我們把髒訂單清乾淨了，存成 `orders_clean.csv`。但單看一張訂單表只看得到數字，看不到**商業洞察**，例如：

- 哪個**地區**的客人買最多？ → 需要把 orders 和 customers join
- 哪個**商品類別**貢獻最多營收？ → 需要把 orders 和 products join
- 每個地區的 Top 3 熱銷商品？ → 需要 groupby + 排序

> **DE/DA 視角**：如果你會 SQL，本節你會很爽；如果不會，本節學完你就等於會了 80% 的 SQL。

**三大工具**：
1. **`merge`** = SQL 的 JOIN（合併多表）
2. **`groupby`** = SQL 的 GROUP BY（分組統計）
3. **`pivot_table`** = Excel 的樞紐分析表（二維交叉統計）


In [2]:
import pandas as pd
import numpy as np

DATA = '../datasets/ecommerce'
orders    = pd.read_csv(f'{DATA}/orders_clean.csv', parse_dates=['order_date'])
customers = pd.read_csv(f'{DATA}/customers.csv')
products  = pd.read_csv(f'{DATA}/products.csv')

print('orders   :', orders.shape,    list(orders.columns))
print('customers:', customers.shape, list(customers.columns))
print('products :', products.shape,  list(products.columns))

orders   : (188, 6) ['order_id', 'customer_id', 'product_id', 'qty', 'order_date', 'amount']
customers: (26, 5) ['customer_id', 'customer_name', 'region', 'signup_date', 'vip_level']
products : (30, 5) ['product_id', 'product_name', 'category', 'unit_price', 'stock_qty']


In [3]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from common.checker import check

---
## 核心觀念 1／3 — 條件篩選（複習 + 加強）

S2 學過 `df.loc[mask]`，本節加強兩招：`isin` 和 `between`。


In [4]:
# 1) 找出金額在 1000~5000 之間的訂單
mid = orders[orders['amount'].between(1000, 5000)]
print('中間金額訂單:', len(mid))

# 2) 找出指定顧客的訂單（用 isin 一次篩多個值）
vip_ids = [2001, 2005, 2010]
vip_orders = orders[orders['customer_id'].isin(vip_ids)]
print('VIP 訂單:', len(vip_orders))

# 3) 複合條件：高金額 + 特定顧客
hit = orders[(orders['amount'] > 3000) & (orders['customer_id'].isin(vip_ids))]
print('複合條件命中:', len(hit))

中間金額訂單: 94
VIP 訂單: 19
複合條件命中: 11


**口訣**：`between` 找區間、`isin` 找清單、`&/|` 組合條件（記得加括號）。


---
## 核心觀念 2／3 — `groupby`：分組 + 聚合

**SQL 對照**：
```sql
SELECT customer_id, SUM(amount) 
FROM orders 
GROUP BY customer_id;
```
↓ 轉成 Pandas ↓
```python
orders.groupby('customer_id')['amount'].sum()
```


In [5]:
# 1) 每個顧客的總消費: groupby()[].sum()
per_customer = orders.groupby('customer_id')['amount'].sum()
print(per_customer.head())
print('型別:', type(per_customer).__name__)  # Series

customer_id
2001    16095.0
2002    16211.0
2003    23007.0
2004    39085.0
2005    34917.0
Name: amount, dtype: float64
型別: Series


In [6]:
# 2) 多種聚合一次做
summary = orders.groupby('customer_id')['amount'].agg(['count', 'sum', 'mean'])
summary.columns = ['訂單數', '總金額', '平均金額']
print(summary.head())

# 3) 排序找 Top 5: sort_values()，ascending=False 代表由大到小
print('\n💰 總金額 Top 5 顧客:')
print(summary.sort_values('總金額', ascending=False).head())

             訂單數      總金額         平均金額
customer_id                           
2001           3  16095.0  5365.000000
2002           7  16211.0  2315.857143
2003          10  23007.0  2300.700000
2004           7  39085.0  5583.571429
2005           8  34917.0  4364.625000

💰 總金額 Top 5 顧客:
             訂單數      總金額         平均金額
customer_id                           
2015          12  56165.0  4680.416667
2022          13  51242.0  3941.692308
2025          12  46739.0  3894.916667
2018          12  43486.0  3623.833333
2021           9  40346.0  4482.888889


In [7]:
# 4) 多欄聚合：不同欄用不同函式
multi = orders.groupby('customer_id').agg(
    訂單數=('order_id', 'count'),
    總金額=('amount', 'sum'),
    平均金額=('amount', 'mean'),
    最大單筆=('amount', 'max'),
).reset_index() # 給予編號
 
multi.sort_values('總金額', ascending=False).head() # 顧客總金額 Top 5 

,customer_id,訂單數,總金額,平均金額,最大單筆
14,2015,12,56165.0,4680.416667,10435.0
21,2022,13,51242.0,3941.692308,9580.0
24,2025,12,46739.0,3894.916667,9755.0
17,2018,12,43486.0,3623.833333,7530.0
20,2021,9,40346.0,4482.888889,9230.0


**口訣**：`groupby(分組欄).agg(...)` → 一行程式替代 SQL 的整段 GROUP BY。


> ### 💡 知識補給站 — groupby 背後的 Hash Table
> 
> `groupby('customer_id')` 底層做了什麼？Pandas 建了一個 **hash table**（雜湊表），把每個 `customer_id` 映射到它出現的**行號清單**，然後對每組套用聚合函式。
> 
> - 這跟 SQL 的 `GROUP BY` 概念一模一樣 — 資料庫也是用 **hash aggregate** 或 sort aggregate
> - Hash table 的查找效率是 **O(1)**，所以 groupby 整體效能是 **O(n)** — 100 萬筆訂單也能在毫秒內完成
> - 效能瓶頸不在分組數量，而在每組的聚合運算複雜度
> 
> 這也是為什麼 Pandas 的 `groupby` 跟用 `for` 迴圈自己分組比起來快非常多 — 底層是 C 語言實作的 hash table，不是 Python 層級的字典操作。
> 
> *延伸關鍵字：hash table, hash aggregate, O(n) complexity, data structure*

---
## 核心觀念 3／3 — `merge`：合併多表（= SQL JOIN）

四種 join 一張圖記熟：

| `how=` | 意義 | 什麼時候用 |
|---|---|---|
| `'inner'` | 兩邊都有才保留 | 最安全、最常用 |
| `'left'`  | 保留左表全部 | 以主表為準 |
| `'right'` | 保留右表全部 | 少用 |
| `'outer'` | 全部都保留 | 找遺漏用 |


In [8]:
# 1) 兩表 join：orders + customers（看每筆訂單屬於哪個地區）
oc = orders.merge(customers, on='customer_id', how='left')
print('合併後形狀:', oc.shape)
print('新增的欄位:', [c for c in oc.columns if c not in orders.columns])
oc.head()

合併後形狀: (188, 10)
新增的欄位: ['customer_name', 'region', 'signup_date', 'vip_level']


,order_id,customer_id,product_id,qty,order_date,amount,customer_name,region,signup_date,vip_level
0,5064,2022,1026,4.0,2025-03-26,8600.0,Victor Lin,North,2023-02-27,Gold
1,5023,2026,1021,5.0,2025-01-05,1355.0,Zoe Huang,South,2023-05-16,Platinum
2,5123,2013,1013,2.0,2025-09-11,3538.0,Mia Huang,North,2023-07-17,Platinum
3,5118,2005,1028,1.0,2025-05-22,1618.0,Emma Liu,West,2023-05-18,Bronze
4,5161,2017,1019,3.0,2025-08-20,1846.0,Quinn Chen,East,2023-08-11,Silver


In [9]:
# 2) 三表 join：再加上 products
#    注意：orders 裡是 product_id，products 裡也是 product_id → on='product_id'
full = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('三表合併後形狀:', full.shape)
print('欄位:', list(full.columns))
full.head()

三表合併後形狀: (188, 14)
欄位: ['order_id', 'customer_id', 'product_id', 'qty', 'order_date', 'amount', 'customer_name', 'region', 'signup_date', 'vip_level', 'product_name', 'category', 'unit_price', 'stock_qty']


,order_id,customer_id,product_id,qty,order_date,amount,customer_name,region,signup_date,vip_level,product_name,category,unit_price,stock_qty
0,5064,2022,1026,4.0,2025-03-26,8600.0,Victor Lin,North,2023-02-27,Gold,Dumbbell 5kg,Sports,2150,51
1,5023,2026,1021,5.0,2025-01-05,1355.0,Zoe Huang,South,2023-05-16,Platinum,Throw Pillow,Home,271,150
2,5123,2013,1013,2.0,2025-09-11,3538.0,Mia Huang,North,2023-07-17,Platinum,Cotton T-Shirt,Clothing,1769,174
3,5118,2005,1028,1.0,2025-05-22,1618.0,Emma Liu,West,2023-05-18,Bronze,Water Bottle,Sports,1618,186
4,5161,2017,1019,3.0,2025-08-20,1846.0,Quinn Chen,East,2023-08-11,Silver,Coffee Mug,Home,1846,274


**口訣**：`merge(另一張表, on='共同欄', how='left/inner')` — 從左表的角度去黏上右表的資訊。


> ### 💡 知識補給站 — merge 的隱藏陷阱：validate 參數
> 
> `merge` 預設**不檢查 key 的唯一性**。如果 `products.csv` 裡同一個 `product_id` 出現兩次，merge 後列數會**暴增**（笛卡爾積），而且**不會報錯** — 這是真實工作中最常見的 silent data quality bug。
> 
> 解法：加上 `validate` 參數，key 不符預期會直接噴 `MergeError`：
> ```python
> orders.merge(products, on='product_id', how='left', validate='many_to_one')
> ```
> 
> | validate 值 | 意義 |
> |---|---|
> | `'one_to_one'` | 兩邊的 key 都必須唯一 |
> | `'many_to_one'` | 右表的 key 必須唯一（最常用） |
> | `'one_to_many'` | 左表的 key 必須唯一 |
> 
> 類比：結婚登記處會檢查「雙方都是未婚」— 不檢查就會出問題。
> 
> **實務建議**：每次 merge 完第一件事就是 `print(df.shape)` 確認列數是否合理。
> 
> *延伸關鍵字：merge validate, Cartesian product, data quality, join explosion*

---
## 實務 Case — 電商銷售洞察分析

**情境**：老闆下午開會要你回答三個問題：
1. 🏆 各**地區**的總營收排名？
2. 🔥 各地區的 **Top 3 熱銷商品**是什麼？
3. 💎 **VIP 等級**的營收貢獻佔比？


In [10]:
# 先把三表合併起來當分析基底
df = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('分析基底:', df.shape)
df[['order_id','customer_name','region','vip_level','product_name','category','amount']].head()

分析基底: (188, 14)


,order_id,customer_name,region,vip_level,product_name,category,amount
0,5064,Victor Lin,North,Gold,Dumbbell 5kg,Sports,8600.0
1,5023,Zoe Huang,South,Platinum,Throw Pillow,Home,1355.0
2,5123,Mia Huang,North,Platinum,Cotton T-Shirt,Clothing,3538.0
3,5118,Emma Liu,West,Bronze,Water Bottle,Sports,1618.0
4,5161,Quinn Chen,East,Silver,Coffee Mug,Home,1846.0


In [11]:
# Q1: 各地區總營收排名
region_rev = df.groupby('region')['amount'].sum().sort_values(ascending=False)
print('🏆 地區營收排名:')
print(region_rev)

🏆 地區營收排名:
region
South    272603.0
North    250449.0
West     110248.0
East      53088.0
Name: amount, dtype: float64


In [12]:
# Q2: 各地區 Top 3 熱銷商品（by 銷售金額）
#     技巧：先 groupby 兩層 → sort → 每個地區取前 3
product_by_region = (
    df.groupby(['region', 'product_name'])['amount']
      .sum()
      .reset_index()
      .sort_values(['region', 'amount'], ascending=[True, False])
)

top3 = product_by_region.groupby('region').head(3)
print('🔥 各地區 Top 3 熱銷商品:')
print(top3)

🔥 各地區 Top 3 熱銷商品:
   region    product_name   amount
2    East      Clean Code   9580.0
1    East      Candle Set   8348.0
13   East       Webcam HD   6872.0
17  North      Clean Code  30656.0
35  North  Statistics 101  24096.0
37  North    Water Bottle  19416.0
48  South          Hoodie  33544.0
65  South        Yoga Mat  31216.0
62  South    Water Bottle  21034.0
66   West      Candle Set  22957.0
71   West    Dumbbell 5kg  15050.0
72   West          Hoodie  11980.0


In [13]:
# Q3: VIP 等級貢獻佔比（用 pivot_table 或 groupby 都可以）
vip_rev = df.groupby('vip_level')['amount'].sum()
vip_pct = (vip_rev / vip_rev.sum() * 100).round(1) # 取百分比

vip_report = pd.DataFrame({'總金額': vip_rev, '佔比(%)': vip_pct})
vip_report = vip_report.sort_values('總金額', ascending=False)
print('💎 VIP 等級貢獻:')
print(vip_report)

💎 VIP 等級貢獻:
                總金額  佔比(%)
vip_level                 
Gold       285982.0   41.7
Platinum   171654.0   25.0
Bronze     151339.0   22.0
Silver      77413.0   11.3


In [14]:
# Bonus: pivot_table — 交叉看「地區 × 品類」的營收矩陣
pivot = df.pivot_table(
    index='region',
    columns='category',
    values='amount',
    aggfunc='sum',
    fill_value=0,
)
print('📊 地區 × 品類 營收交叉表:')
print(pivot)

📊 地區 × 品類 營收交叉表:
category    Books  Clothing  Electronics     Home   Sports
region                                                    
East      15728.0    2396.0       8889.0  12153.0  13922.0
North     83330.0   43390.0      46345.0  30919.0  46465.0
South     55698.0   70307.0      31718.0  26449.0  88431.0
West      27488.0   17748.0      13283.0  24232.0  27497.0


In [15]:
# 存一份合併好的分析基底給 S4/S5/S6 用
df.to_csv('../datasets/ecommerce/orders_enriched.csv', index=False)
print('已存 orders_enriched.csv，供後續 session 使用')

已存 orders_enriched.csv，供後續 session 使用


---
## 課堂練習（40 min）

### 🟢 送分題
用 `orders` 算出：
1. 每個 `customer_id` 的訂單總數


In [34]:
# TODO
customer_order = orders.groupby('customer_id')['order_id'].count().head()
print(customer_order)

customer_id
2001     3
2002     7
2003    10
2004     7
2005     8
Name: order_id, dtype: int64


### 🟡 核心題
用三表合併的 `df`，回答：
1. 哪個**商品類別**總營收最高？
2. **Gold** 等級的顧客一共下了多少筆訂單？金額多少？
3. 各**地區**的平均訂單金額是多少？


In [31]:
# TODO
top_product = df.groupby('category')['amount'].sum().sort_values(ascending=False)
print(f"總營收最高的商品: {top_product.index[0]}，金額: {top_product.values[0]:.2f}")

gold_level = df[df['vip_level'] == 'Gold']
print(f"Gold等級一共下訂了 {len(gold_level)} 筆訂單，總營收為 {gold_level['amount'].sum():.2f}")

region_average = df.groupby('region')['amount']
for region, avg in region_average:
    print(f"地區: {region}，平均訂單金額: {avg.mean():.2f}")


總營收最高的商品: Hoodie，金額: 59900.00
Gold等級一共下訂了 79 筆訂單，總營收為 285982.00
地區: East，平均訂單金額: 3318.00
地區: North，平均訂單金額: 3738.04
地區: South，平均訂單金額: 3786.15
地區: West，平均訂單金額: 3340.85


### 🔴 挑戰題
計算每位顧客的 **RFM 粗估**：
- **R**ecency：最近一次下單日期（用 `max(order_date)`）
- **F**requency：下單次數
- **M**onetary：總金額

最後用 M 排序，列出 Top 5 顧客（加上顧客名字）。

> 這是電商最經典的顧客分群指標，會做這題你就有 junior DA 實力了。


In [48]:
# TODO
# 計算 RFM
recency = df.groupby('customer_id')['order_date'].max()
frequency = df.groupby('customer_id')['order_id'].count()
monetary = df.groupby('customer_id')['amount'].sum()    

# 合併成 DataFrame
rfm = pd.DataFrame({
    'customer_id': recency.index,
    'Recency': recency.values,
    'Frequency': frequency.values,
    'Monetary': monetary.values
})

# 加上顧客名字
rfm = rfm.merge(customers[['customer_id', 'customer_name']], on='customer_id', how='left')

# 按 Monetary 排序，取 Top 5
top5 = rfm.sort_values('Monetary', ascending=False).head(5)
print('Top 5 顧客 by Monetary:')
print(top5)



Top 5 顧客 by Monetary:
    customer_id    Recency  Frequency  Monetary customer_name
14         2015 2025-12-23         12   56165.0  Olivia Huang
21         2022 2025-11-27         13   51242.0    Victor Lin
24         2025 2025-11-27         12   46739.0     Yuki Chen
17         2018 2025-12-26         12   43486.0    Rachel Lin
20         2021 2025-12-31          9   40346.0      Uma Wang


In [47]:
# 🏁 大地遊戲 — 哪個品類總營收最高？答對就能找到解答！
top_category = df.groupby('category')['amount'].sum().idxmax()
check('S3', top_category)



🎉 答對了！解答在這裡 → 📂 docs/_archive/S3_ref.ipynb


---
## 小結與速查表

| 想做什麼 | 怎麼寫 | SQL 對照 |
|---|---|---|
| 區間篩選 | `df[df['x'].between(a,b)]` | `WHERE x BETWEEN a AND b` |
| 清單篩選 | `df[df['x'].isin([...])]` | `WHERE x IN (...)` |
| 分組加總 | `df.groupby('k')['v'].sum()` | `GROUP BY k` |
| 多聚合 | `df.groupby('k').agg(...)` | 多 `SUM/COUNT/AVG` |
| Join 兩表 | `a.merge(b, on='k', how='left')` | `LEFT JOIN` |
| 樞紐分析 | `df.pivot_table(index, columns, values)` | (Excel 樞紐) |
| 每組 Top N | `sort + groupby.head(n)` | `ROW_NUMBER()` |

**下節預告 S4**：我們有 `order_date`，但還沒用它做時序分析。下節會教你用 1 行程式算出「月度營收」「7 日移動平均」「年月週趨勢」。
